# Testing the EURUSD:

### AUDUSD NZDUSD USDZAR EURSEK EURNOK

### Imports

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# 1. Define the Datasets
datasets = {
    'AUDUSD': ['202601', '202602', '202603'],
    'NZDUSD': ['202601', '202602', '202603'],
    'USDZAR': ['202511', '202512', '202601'],
    'EURSEK': ['202601', '202602', '202603'],
    'EURNOK': ['202601', '202602', '202603']
}

# --- THE FIX: Lower bar requirements for less liquid pairs ---
TRAIN_BARS = 1000
TEST_BARS = 250
TICK_WINDOW = 2500

final_leaderboard = []

print("🌍 Starting Multi-Asset Robustness Tournament (Low-Liquidity Adjusted)...\n")

for pair, months in datasets.items():
    print(f"========================================")
    print(f"🚀 Processing {pair}...")
    print(f"========================================")
    
    # --- A. Data Loading & Compilation ---
    bars_list = []
    for m in months:
        try:
            bid = pd.read_parquet(f"../data/processed/{pair.lower()}_dukascopy_bid_{m}.parquet")
            ask = pd.read_parquet(f"../data/processed/{pair.lower()}_dukascopy_ask_{m}.parquet")
            
            bid['datetime'] = pd.to_datetime(bid['datetime'])
            ask['datetime'] = pd.to_datetime(ask['datetime'])
            bid = bid.sort_values('datetime')
            ask = ask.sort_values('datetime')
            
            merged = pd.merge_asof(
                bid, 
                ask[['datetime', 'price']], 
                on='datetime', 
                direction='backward', 
                suffixes=('_bid', '_ask')
            )
            merged['mid_price'] = (merged['price_bid'] + merged['price_ask']) / 2
            
            # Sample and store
            sampled = merged.iloc[::TICK_WINDOW].copy()
            bars_list.append(sampled)
        except Exception as e:
            print(f"  [!] Missing data for {pair} {m}: {e}")
            continue
            
    if not bars_list:
        continue
        
    df_bars = pd.concat(bars_list).reset_index(drop=True)
    df_bars['log_returns'] = np.log(df_bars['mid_price'] / df_bars['mid_price'].shift(1))
    df_bars.dropna(inplace=True)
    df_bars['vol_proxy'] = np.abs(df_bars['log_returns']) * 1e6
    
    total_bars = len(df_bars)
    print(f"  Total {TICK_WINDOW}-tick bars compiled: {total_bars}")
    
    # --- B. Rolling Bar-Block Walk-Forward ---
    pair_results = []
    start_idx = 0
    fold = 1
    
    while (start_idx + TRAIN_BARS + TEST_BARS) <= total_bars:
        train_data = df_bars['vol_proxy'].iloc[start_idx : start_idx + TRAIN_BARS]
        test_data = df_bars['vol_proxy'].iloc[start_idx + TRAIN_BARS : start_idx + TRAIN_BARS + TEST_BARS]
        full_data = pd.concat([train_data, test_data])
        
        # 1. Baseline: SMA of last 5 bars
        sma_forecast = full_data.rolling(window=5).mean().shift(1).iloc[TRAIN_BARS:]
        sma_rmse = np.sqrt(mean_squared_error(test_data, sma_forecast.fillna(method='bfill')))
        
        # 2. AR(1)
        ar_model = AutoReg(train_data.values, lags=1).fit()
        ar_forecast = ar_model.predict(start=TRAIN_BARS, end=TRAIN_BARS + TEST_BARS - 1, dynamic=False)
        ar_rmse = np.sqrt(mean_squared_error(test_data.values, ar_forecast))
        
        # 3. MS-AR(1)
        try:
            ms_model = sm.tsa.MarkovAutoregression(
                endog=full_data.values, 
                k_regimes=2, 
                order=1, 
                switching_ar=False, 
                switching_variance=True
            )
            ms_res = ms_model.fit(method='bfgs', disp=False)
            ms_forecast = ms_res.fittedvalues[-TEST_BARS:]
            ms_rmse = np.sqrt(mean_squared_error(test_data.values, ms_forecast))
        except Exception:
            ms_rmse = np.nan
            
        pair_results.append({
            'Fold': fold,
            'SMA_RMSE': sma_rmse,
            'AR_RMSE': ar_rmse,
            'MSAR_RMSE': ms_rmse
        })
        
        print(f"  Fold {fold} complete.")
        start_idx += TEST_BARS  # Roll forward by the test window size
        fold += 1
        
    # --- C. Aggregate Pair Results ---
    if pair_results:
        res_df = pd.DataFrame(pair_results)
        avg_sma = res_df['SMA_RMSE'].mean()
        avg_ar = res_df['AR_RMSE'].mean()
        avg_msar = res_df['MSAR_RMSE'].mean()
        
        # Calculate how much better MS-AR is than standard AR
        edge_pct = ((avg_ar - avg_msar) / avg_ar) * 100
        
        final_leaderboard.append({
            'Pair': pair,
            'Folds Tested': len(res_df),
            'SMA RMSE': round(avg_sma, 2),
            'AR(1) RMSE': round(avg_ar, 2),
            'MS-AR(1) RMSE': round(avg_msar, 2),
            'MS-AR Edge (%)': round(edge_pct, 1)
        })

# --- D. Print Final Leaderboard ---
print("\n" + "="*70)
print("🏆 MULTI-ASSET ROBUSTNESS LEADERBOARD (AVERAGE RMSE)")
print("="*70)
leader_df = pd.DataFrame(final_leaderboard).sort_values(by='MS-AR Edge (%)', ascending=False)
print(leader_df.to_string(index=False))

🌍 Starting Multi-Asset Robustness Tournament (Low-Liquidity Adjusted)...

🚀 Processing AUDUSD...
  Total 2500-tick bars compiled: 2515
  Fold 1 complete.
  Fold 2 complete.
  Fold 3 complete.
  Fold 4 complete.
  Fold 5 complete.
  Fold 6 complete.
🚀 Processing NZDUSD...
  Total 2500-tick bars compiled: 1768
  Fold 1 complete.
  Fold 2 complete.
  Fold 3 complete.
🚀 Processing USDZAR...
  Total 2500-tick bars compiled: 4323
  Fold 1 complete.
  Fold 2 complete.
  Fold 3 complete.
  Fold 4 complete.
  Fold 5 complete.
  Fold 6 complete.
  Fold 7 complete.
  Fold 8 complete.
  Fold 9 complete.
  Fold 10 complete.
  Fold 11 complete.
  Fold 12 complete.
  Fold 13 complete.
🚀 Processing EURSEK...
  Total 2500-tick bars compiled: 2034
  Fold 1 complete.
  Fold 2 complete.
  Fold 3 complete.
  Fold 4 complete.
🚀 Processing EURNOK...
  Total 2500-tick bars compiled: 3421
  Fold 1 complete.
  Fold 2 complete.
  Fold 3 complete.
  Fold 4 complete.
  Fold 5 complete.
  Fold 6 complete.
  Fold 7 